# Notebook 08 — Freeze the Train-Only Raw-Rank Signal Policy

This notebook does **not** open or evaluate the unseen test.

Stage 06 already selected:

- model family: XGBoost;
- feature set: `I_full_40`;
- trial: the frozen Stage 06 selected trial;
- policy: daily Top 5%, minimum one signal per date;
- score use: raw cross-sectional ranking only.

Notebook 08 therefore performs no new model selection, calibration search,
probability-threshold search, or signal-fraction search. It reconstructs the
exact Stage 06 out-of-fold policy for the exact selected model and feature set,
verifies identity with the Stage 06 decision, and freezes the downstream policy.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


Python: 3.12.2
numpy: 1.26.4
pandas: 2.2.3


## 1. Locate the repository and import the exact Stage 06 policy implementation


In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Repository root was not found. "
        "Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.models.policy_selection import (
    POLICY_SELECTION_SCHEMA_VERSION,
    DailyTopFractionPolicy,
    apply_daily_top_fraction_policy,
    summarize_policy_predictions,
)
from src.utils.paths import repository_result_paths
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


e:\Iran_stock_trade\financial-ml-decision-support\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load frozen Stage 06 and Stage 07 lineage


In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(
            f"Configuration must be a mapping: {path}"
        )

    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(
            f"JSON artifact must be an object: {path}"
        )

    return value


paths_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "paths.yaml"
)
policy_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "signal_policy.yaml"
)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

prediction_path = (
    RESULT_PATHS["predictions"]
    / "06_walk_forward_validation_predictions.csv"
)
stage06_decision_path = (
    RESULT_PATHS["manifests"]
    / "06_model_selection_decision.json"
)
stage06_manifest_path = (
    RESULT_PATHS["manifests"]
    / "06_optuna_model_selection_manifest.json"
)
stage07_manifest_path = (
    RESULT_PATHS["manifests"]
    / "07_frozen_model_training_manifest.json"
)

for required_path in [
    prediction_path,
    stage06_decision_path,
    stage06_manifest_path,
    stage07_manifest_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required frozen artifact is missing: "
            f"{required_path}"
        )

stage06_decision = load_json(stage06_decision_path)
stage06_manifest = load_json(stage06_manifest_path)
stage07_manifest = load_json(stage07_manifest_path)

SELECTED_MODEL = str(
    stage06_decision["primary_selected_model"]
)
SELECTED_FEATURE_SET = str(
    stage06_decision[
        "primary_selected_feature_set"
    ]
)
SELECTED_FEATURES = list(
    stage06_decision[
        "primary_selected_features"
    ]
)
SELECTED_TRIAL = int(
    stage06_decision[
        "selected_trial_numbers"
    ][SELECTED_MODEL]
)

fixed_policy = stage06_decision["fixed_policy"]
POLICY_FRACTION = float(
    fixed_policy["selected_fraction"]
)
POLICY_MINIMUM_PER_DATE = int(
    fixed_policy["minimum_signals_per_date"]
)

PRIMARY_POLICY = DailyTopFractionPolicy(
    fraction=POLICY_FRACTION,
    minimum_signals_per_date=(
        POLICY_MINIMUM_PER_DATE
    ),
)

assert policy_config["status"] == (
    "stage_08_configured_v2_stage06_policy_lock"
)
assert POLICY_SELECTION_SCHEMA_VERSION == (
    policy_config["policy"][
        "implementation_schema_version"
    ]
)

assert stage06_manifest["status"] == (
    "frozen_after_integrity_checks"
)
assert stage06_manifest["unseen_test_used"] is False
assert stage06_decision["threshold_selected"] is False
assert stage06_decision[
    "hard_started_filter_applied"
] is False
assert stage06_decision[
    "hard_breadth_filter_applied"
] is False
assert stage06_decision["unseen_test_used"] is False

assert stage07_manifest["status"] == (
    "frozen_after_integrity_checks"
)
assert stage07_manifest["unseen_test_used"] is False
assert stage07_manifest[
    "hard_started_filter_applied"
] is False
assert stage07_manifest[
    "hard_breadth_filter_applied"
] is False
assert stage07_manifest[
    "probability_policy"
]["calibration_fitted"] is False
assert stage07_manifest[
    "probability_policy"
]["threshold_selected"] is False

assert SELECTED_MODEL == (
    policy_config["input"]["selected_model"]
)
assert SELECTED_FEATURE_SET == (
    policy_config["input"][
        "selected_feature_set"
    ]
)
assert SELECTED_MODEL == "xgboost"
assert SELECTED_FEATURE_SET == "I_full_40"
assert len(SELECTED_FEATURES) == 40

assert stage07_manifest["model"][
    "selected_model"
] == SELECTED_MODEL
assert stage07_manifest["model"][
    "selected_feature_set"
] == SELECTED_FEATURE_SET
assert int(
    stage07_manifest["model"][
        "selected_trial_number"
    ]
) == SELECTED_TRIAL
assert list(
    stage07_manifest["features"][
        "model_features"
    ]
) == SELECTED_FEATURES

assert np.isclose(
    POLICY_FRACTION,
    float(
        policy_config["policy"][
            "selected_fraction"
        ]
    ),
)
assert POLICY_MINIMUM_PER_DATE == int(
    policy_config["policy"][
        "minimum_signals_per_date"
    ]
)
assert np.isclose(POLICY_FRACTION, 0.05)
assert POLICY_MINIMUM_PER_DATE == 1

print("Selected model:", SELECTED_MODEL)
print(
    "Selected feature set:",
    SELECTED_FEATURE_SET,
)
print("Selected trial:", SELECTED_TRIAL)
print(
    "Selected raw features:",
    len(SELECTED_FEATURES),
)
print(
    "Frozen policy:",
    f"Top {POLICY_FRACTION:.0%}, "
    f"minimum {POLICY_MINIMUM_PER_DATE}",
)
print(
    "Stage 07 model SHA256:",
    stage07_manifest["model"]["model_sha256"],
)


Selected model: xgboost
Selected feature set: I_full_40
Selected trial: 10
Selected raw features: 40
Frozen policy: Top 5%, minimum 1
Stage 07 model SHA256: 73a781551cdabd6bb67f5b9c0836a8683372e46487d9aade29e6320006086c1f


## 3. Load only the exact selected model and selected feature set OOF predictions


In [4]:
prediction_columns = [
    "model_name",
    "feature_set_name",
    "fold_id",
    "event_id",
    "symbol",
    "dEven",
    "event_end_date",
    "meta_label",
    "probability_positive",
    "validation_metric_weight",
]

all_oof = pd.read_csv(
    prediction_path,
    usecols=prediction_columns,
    low_memory=False,
)

oof = all_oof.loc[
    all_oof["model_name"].eq(SELECTED_MODEL)
    & all_oof["feature_set_name"].eq(
        SELECTED_FEATURE_SET
    )
].copy()

oof["fold_id"] = pd.to_numeric(
    oof["fold_id"],
    errors="raise",
).astype(int)
oof["dEven"] = pd.to_datetime(
    oof["dEven"],
    errors="raise",
).dt.normalize()
oof["event_end_date"] = pd.to_datetime(
    oof["event_end_date"],
    errors="raise",
).dt.normalize()
oof["meta_label"] = pd.to_numeric(
    oof["meta_label"],
    errors="raise",
).astype(int)
oof["probability_positive"] = pd.to_numeric(
    oof["probability_positive"],
    errors="raise",
).astype(float)
oof["validation_metric_weight"] = pd.to_numeric(
    oof["validation_metric_weight"],
    errors="raise",
).astype(float)

oof = oof.sort_values(
    ["fold_id", "dEven", "symbol", "event_id"],
    kind="stable",
).reset_index(drop=True)

expected_rows = int(
    policy_config["input"]["expected_oof_rows"]
)
expected_fold_counts = {
    int(key): int(value)
    for key, value in policy_config[
        "input"
    ]["expected_fold_counts"].items()
}
actual_fold_counts = (
    oof.groupby("fold_id")
    .size()
    .astype(int)
    .to_dict()
)

assert len(oof) == expected_rows
assert actual_fold_counts == expected_fold_counts
assert oof["event_id"].is_unique
assert oof["meta_label"].isin([0, 1]).all()
assert np.isfinite(
    oof["probability_positive"].to_numpy(
        dtype=float
    )
).all()
assert oof["probability_positive"].between(
    0.0,
    1.0,
).all()
assert oof[
    "validation_metric_weight"
].eq(1.0).all()
assert set(oof["fold_id"]) == {1, 2, 3, 4, 5}
assert oof["model_name"].eq(
    SELECTED_MODEL
).all()
assert oof["feature_set_name"].eq(
    SELECTED_FEATURE_SET
).all()

oof_input_audit_df = (
    oof.groupby("fold_id", as_index=False)
    .agg(
        events=("event_id", "size"),
        dates=("dEven", "nunique"),
        first_signal_date=("dEven", "min"),
        last_signal_date=("dEven", "max"),
        positive_fraction=("meta_label", "mean"),
        minimum_raw_score=(
            "probability_positive",
            "min",
        ),
        mean_raw_score=(
            "probability_positive",
            "mean",
        ),
        maximum_raw_score=(
            "probability_positive",
            "max",
        ),
    )
)
oof_input_audit_df[
    "selected_model"
] = SELECTED_MODEL
oof_input_audit_df[
    "selected_feature_set"
] = SELECTED_FEATURE_SET
oof_input_audit_df[
    "selected_trial"
] = SELECTED_TRIAL
oof_input_audit_df[
    "duplicate_event_ids"
] = 0
oof_input_audit_df[
    "nonfinite_scores"
] = 0
oof_input_audit_df[
    "calibration_fitted"
] = False
oof_input_audit_df[
    "threshold_selected"
] = False
oof_input_audit_df[
    "unseen_test_used"
] = False

oof_input_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_oof_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(oof_input_audit_df)


,fold_id,events,dates,first_signal_date,last_signal_date,positive_fraction,minimum_raw_score,mean_raw_score,maximum_raw_score,selected_model,selected_feature_set,selected_trial,duplicate_event_ids,nonfinite_scores,calibration_fitted,threshold_selected,unseen_test_used
0,1,10344,92,2017-09-19,2018-01-31,0.385731,0.297799,0.542050,0.722038,xgboost,I_full_40,10,0,0,False,False,False
1,2,10408,91,2018-02-03,2018-06-27,0.466660,0.308411,0.526818,0.732247,xgboost,I_full_40,10,0,0,False,False,False
2,3,10397,136,2018-06-30,2019-01-15,0.726363,0.273232,0.461357,0.750240,xgboost,I_full_40,10,0,0,False,False,False
3,4,10319,231,2019-01-16,2020-01-04,0.888071,0.303482,0.529864,0.834761,xgboost,I_full_40,10,0,0,False,False,False
4,5,10372,288,2020-01-05,2021-03-14,0.507135,0.329853,0.588739,0.830716,xgboost,I_full_40,10,0,0,False,False,False


## 4. Reconstruct the exact frozen Stage 06 daily policy

No candidate fraction is searched. No probability calibration or threshold is
fitted. The raw XGBoost output is used only as a same-day ranking score.


In [5]:
policy_assignments_df = (
    apply_daily_top_fraction_policy(
        oof,
        policy=PRIMARY_POLICY,
    )
)

policy_summary, policy_fold_metrics_df = (
    summarize_policy_predictions(
        policy_assignments_df,
        policy=PRIMARY_POLICY,
    )
)

expected_true_positive = int(
    stage06_decision[
        "selected_true_positive"
    ]
)
expected_false_positive = int(
    stage06_decision[
        "selected_false_positive"
    ]
)
expected_signals = (
    expected_true_positive
    + expected_false_positive
)
expected_precision = float(
    stage06_decision[
        "selected_precision"
    ]
)
expected_specificity = float(
    stage06_decision[
        "selected_specificity"
    ]
)

assert bool(
    policy_summary["policy_complete"]
)
assert int(
    policy_summary["signals"]
) == expected_signals
assert int(
    policy_summary["true_positive"]
) == expected_true_positive
assert int(
    policy_summary["false_positive"]
) == expected_false_positive
assert np.isclose(
    float(policy_summary["precision"]),
    expected_precision,
    atol=1.0e-12,
    rtol=1.0e-12,
)
assert np.isclose(
    float(policy_summary["specificity"]),
    expected_specificity,
    atol=1.0e-12,
    rtol=1.0e-12,
)

expected_fold_signal_counts = {
    int(key): int(value)
    for key, value in policy_config[
        "policy"
    ]["expected_fold_signal_counts"].items()
}
actual_fold_signal_counts = (
    policy_fold_metrics_df
    .set_index("fold_id")["signals"]
    .astype(int)
    .to_dict()
)

assert (
    actual_fold_signal_counts
    == expected_fold_signal_counts
)
assert int(
    policy_assignments_df[
        "selected_signal"
    ].sum()
) == int(
    policy_config["policy"][
        "expected_total_oof_signals"
    ]
)

date_audit_df = (
    policy_assignments_df.groupby(
        "dEven",
        as_index=False,
    )
    .agg(
        fold_id=("fold_id", "first"),
        candidate_events=("event_id", "size"),
        required_quota=(
            "daily_signal_quota",
            "first",
        ),
        selected_signals=(
            "selected_signal",
            "sum",
        ),
        selected_positive_labels=(
            "meta_label",
            lambda series: int(
                series[
                    policy_assignments_df.loc[
                        series.index,
                        "selected_signal",
                    ].to_numpy(dtype=bool)
                ].sum()
            ),
        ),
        raw_score_cutoff=(
            "daily_score_cutoff",
            "first",
        ),
        minimum_raw_score=(
            "probability_positive",
            "min",
        ),
        maximum_raw_score=(
            "probability_positive",
            "max",
        ),
    )
)

assert date_audit_df[
    "selected_signals"
].eq(
    date_audit_df["required_quota"]
).all()
assert date_audit_df[
    "selected_signals"
].ge(1).all()

policy_summary_df = pd.DataFrame(
    [
        {
            key: value
            for key, value in (
                policy_summary.items()
            )
            if key != "policy"
        }
    ]
)
policy_summary_df.insert(
    0,
    "selected_feature_set",
    SELECTED_FEATURE_SET,
)
policy_summary_df.insert(
    0,
    "selected_model",
    SELECTED_MODEL,
)

policy_fold_metrics_df.insert(
    0,
    "selected_feature_set",
    SELECTED_FEATURE_SET,
)
policy_fold_metrics_df.insert(
    0,
    "selected_model",
    SELECTED_MODEL,
)

policy_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
policy_fold_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
date_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_date_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

prediction_output_columns = [
    "model_name",
    "feature_set_name",
    "fold_id",
    "event_id",
    "symbol",
    "dEven",
    "event_end_date",
    "meta_label",
    "probability_positive",
    "daily_candidate_count",
    "daily_rank",
    "daily_signal_quota",
    "daily_score_cutoff",
    "selected_signal",
]

policy_assignments_df[
    prediction_output_columns
].to_csv(
    RESULT_PATHS["predictions"]
    / "08_oof_signal_policy_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(policy_summary_df)
display(policy_fold_metrics_df)

print(
    "Total selected OOF signals:",
    int(policy_summary["signals"]),
)
print(
    "True positives:",
    int(policy_summary["true_positive"]),
)
print(
    "False positives:",
    int(policy_summary["false_positive"]),
)
print(
    "Precision:",
    float(policy_summary["precision"]),
)


,selected_model,selected_feature_set,true_positive,false_positive,true_negative,false_negative,events,signals,expected_signals,signal_rate,...,date_coverage,precision,specificity,sensitivity,prevalence,min_fold_specificity,mean_fold_specificity,std_fold_specificity,min_fold_precision,policy_complete
0,xgboost,I_full_40,1987,1029,19988,28836,51840,3016,3016,0.058179,...,1.0,0.65882,0.95104,0.064465,0.594579,0.944589,0.950631,0.004129,0.440285,True


,selected_model,selected_feature_set,fold_id,true_positive,false_positive,true_negative,false_negative,signals,events,dates,precision,specificity,sensitivity
0,xgboost,I_full_40,1,247,314,6040,3743,561,10344,92,0.440285,0.950582,0.061905
1,xgboost,I_full_40,2,295,271,5280,4562,566,10408,91,0.521201,0.951180,0.060737
2,xgboost,I_full_40,3,473,121,2724,7079,594,10397,136,0.796296,0.957469,0.062632
3,xgboost,I_full_40,4,560,64,1091,8604,624,10319,231,0.897436,0.944589,0.061109
4,xgboost,I_full_40,5,412,259,4853,4848,671,10372,288,0.614009,0.949335,0.078327


Total selected OOF signals: 3016
True positives: 1987
False positives: 1029
Precision: 0.6588196286472149


## 5. Freeze the downstream raw-rank policy and lineage


In [6]:
policy_artifact = {
    "stage": 8,
    "status": "frozen_train_only_policy",
    "stage_version": (
        "v2_stage06_selected_model_feature_set_"
        "raw_rank_policy_lock"
    ),
    "score_policy": {
        "source_model": SELECTED_MODEL,
        "source_feature_set": (
            SELECTED_FEATURE_SET
        ),
        "source_trial": SELECTED_TRIAL,
        "source_model_sha256": (
            stage07_manifest["model"][
                "model_sha256"
            ]
        ),
        "source_prediction_column": (
            "probability_positive"
        ),
        "selected_calibration_method": (
            "raw_identity"
        ),
        "probability_calibrator_fitted": False,
        "score_name_for_downstream_use": (
            "xgboost_ranking_score"
        ),
        "literal_probability_interpretation_allowed": (
            False
        ),
        "ranking_use_allowed": True,
    },
    "feature_lineage": {
        "selected_feature_set": (
            SELECTED_FEATURE_SET
        ),
        "selected_raw_features": (
            SELECTED_FEATURES
        ),
        "selected_raw_feature_count": int(
            len(SELECTED_FEATURES)
        ),
    },
    "signal_policy": {
        "policy_type": (
            "daily_cross_sectional_top_fraction"
        ),
        "implementation_schema_version": (
            POLICY_SELECTION_SCHEMA_VERSION
        ),
        "selected_fraction": (
            POLICY_FRACTION
        ),
        "minimum_signals_per_date": (
            POLICY_MINIMUM_PER_DATE
        ),
        "quota_rounding": "ceil",
        "ranking_order": [
            "score_descending",
            "symbol_ascending",
            "event_id_ascending",
        ],
        "fixed_probability_threshold_selected": (
            False
        ),
        "daily_cutoff_is_dynamic": True,
        "expected_total_oof_signals": int(
            policy_summary["signals"]
        ),
        "fold_signal_counts": (
            actual_fold_signal_counts
        ),
    },
    "selection_evidence": {
        "policy_selected_in_stage06": True,
        "policy_reselected_in_stage08": False,
        "calibration_candidates_evaluated": False,
        "signal_fraction_candidates_evaluated": (
            False
        ),
        "economic_returns_used": False,
        "portfolio_metrics_used": False,
        "unseen_test_used": False,
    },
    "downstream_rule": (
        "For each signal date, score every eligible "
        "long candidate with the frozen Stage 07 "
        "XGBoost I_full_40 pipeline; rank raw scores "
        "descending, then symbol and event_id ascending; "
        "select max(1, ceil(0.05 * candidate_count))."
    ),
}

policy_artifact_path = (
    RESULT_PATHS["manifests"]
    / "08_probability_signal_policy.json"
)
policy_artifact_path.write_text(
    json.dumps(
        policy_artifact,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

manifest = {
    "stage": 8,
    "status": "frozen_after_integrity_checks",
    "stage_version": (
        "v2_stage06_policy_lock_no_reselection"
    ),
    "notebook": (
        "08_unseen_test_evaluation.ipynb"
    ),
    "git_commit_sha": git_commit_sha(
        REPOSITORY_ROOT
    ),
    "software": software_manifest(),
    "lineage": {
        "stage06_code_commit_sha": (
            stage06_manifest[
                "git_commit_sha"
            ]
        ),
        "stage06_configuration_hash": (
            stage06_manifest[
                "configuration_hash"
            ]
        ),
        "stage06_study_signature": (
            stage06_manifest[
                "study_signature"
            ]
        ),
        "stage06_selected_model": (
            SELECTED_MODEL
        ),
        "stage06_selected_feature_set": (
            SELECTED_FEATURE_SET
        ),
        "stage06_selected_trial": (
            SELECTED_TRIAL
        ),
        "stage07_code_commit_sha": (
            stage07_manifest[
                "git_commit_sha"
            ]
        ),
        "stage07_model_sha256": (
            stage07_manifest["model"][
                "model_sha256"
            ]
        ),
        "stage07_training_population_sha256": (
            stage07_manifest[
                "training_population"
            ][
                "training_population_sha256"
            ]
        ),
    },
    "oof_input": {
        "rows": int(len(oof)),
        "fold_counts": actual_fold_counts,
        "first_signal_date": str(
            oof["dEven"].min()
        ),
        "last_signal_date": str(
            oof["dEven"].max()
        ),
        "selected_model": SELECTED_MODEL,
        "selected_feature_set": (
            SELECTED_FEATURE_SET
        ),
    },
    "score_policy": (
        policy_artifact["score_policy"]
    ),
    "signal_policy": (
        policy_artifact["signal_policy"]
    ),
    "oof_policy_metrics": {
        "signals": int(
            policy_summary["signals"]
        ),
        "true_positive": int(
            policy_summary["true_positive"]
        ),
        "false_positive": int(
            policy_summary["false_positive"]
        ),
        "precision": float(
            policy_summary["precision"]
        ),
        "specificity": float(
            policy_summary["specificity"]
        ),
    },
    "safeguards": {
        "policy_reselected": False,
        "calibrator_fitted": False,
        "probability_threshold_selected": (
            False
        ),
        "hard_started_filter_applied": False,
        "hard_breadth_filter_applied": False,
        "unseen_test_loaded": False,
        "unseen_test_labels_used": False,
        "economic_returns_used_for_selection": (
            False
        ),
        "portfolio_metrics_used_for_selection": (
            False
        ),
        "model_retrained": False,
    },
    "configuration_hash": stable_object_hash(
        {
            "policy_config": policy_config,
            "stage06_decision": (
                stage06_decision
            ),
            "stage06_configuration_hash": (
                stage06_manifest[
                    "configuration_hash"
                ]
            ),
            "stage07_model_sha256": (
                stage07_manifest["model"][
                    "model_sha256"
                ]
            ),
            "policy_summary": {
                key: value
                for key, value in (
                    policy_summary.items()
                )
                if key != "policy"
            },
            "policy_artifact": (
                policy_artifact
            ),
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "08_probability_signal_policy_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Policy artifact:", policy_artifact_path)
print("Manifest:", manifest_path)
print(
    "Manifest code SHA:",
    manifest["git_commit_sha"],
)


Policy artifact: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\08_probability_signal_policy.json
Manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\08_probability_signal_policy_manifest.json
Manifest code SHA: 56d2b5e5e2d22f4bd46f8a62a058e09a26531b31


## 6. Final integrity checks


In [7]:
assert SELECTED_MODEL == "xgboost"
assert SELECTED_FEATURE_SET == "I_full_40"
assert SELECTED_TRIAL == int(
    stage07_manifest["model"][
        "selected_trial_number"
    ]
)
assert len(SELECTED_FEATURES) == 40

assert len(oof) == 51_840
assert oof["event_id"].is_unique
assert actual_fold_counts == {
    1: 10_344,
    2: 10_408,
    3: 10_397,
    4: 10_319,
    5: 10_372,
}

assert np.isclose(
    POLICY_FRACTION,
    0.05,
)
assert POLICY_MINIMUM_PER_DATE == 1
assert int(
    policy_summary["signals"]
) == 3_016
assert int(
    policy_summary["true_positive"]
) == 1_987
assert int(
    policy_summary["false_positive"]
) == 1_029
assert np.isclose(
    float(policy_summary["precision"]),
    0.6588196286472149,
    atol=1.0e-12,
    rtol=1.0e-12,
)

assert actual_fold_signal_counts == {
    1: 561,
    2: 566,
    3: 594,
    4: 624,
    5: 671,
}
assert date_audit_df[
    "selected_signals"
].eq(
    date_audit_df["required_quota"]
).all()
assert date_audit_df[
    "selected_signals"
].ge(1).all()

assert policy_artifact[
    "score_policy"
][
    "probability_calibrator_fitted"
] is False
assert policy_artifact[
    "score_policy"
][
    "literal_probability_interpretation_allowed"
] is False
assert policy_artifact[
    "signal_policy"
][
    "fixed_probability_threshold_selected"
] is False
assert policy_artifact[
    "selection_evidence"
][
    "policy_reselected_in_stage08"
] is False

assert manifest["safeguards"][
    "policy_reselected"
] is False
assert manifest["safeguards"][
    "calibrator_fitted"
] is False
assert manifest["safeguards"][
    "probability_threshold_selected"
] is False
assert manifest["safeguards"][
    "hard_started_filter_applied"
] is False
assert manifest["safeguards"][
    "hard_breadth_filter_applied"
] is False
assert manifest["safeguards"][
    "unseen_test_loaded"
] is False
assert manifest["safeguards"][
    "unseen_test_labels_used"
] is False
assert manifest["safeguards"][
    "economic_returns_used_for_selection"
] is False
assert manifest["safeguards"][
    "portfolio_metrics_used_for_selection"
] is False
assert manifest["safeguards"][
    "model_retrained"
] is False

print("Notebook 08 integrity checks: PASSED")
print("Selected model:", SELECTED_MODEL)
print(
    "Selected feature set:",
    SELECTED_FEATURE_SET,
)
print("Selected trial:", SELECTED_TRIAL)
print(
    "Raw feature count:",
    len(SELECTED_FEATURES),
)
print("Score policy: raw identity ranking")
print("Probability calibrator fitted: False")
print(
    "Literal probability interpretation allowed: False"
)
print(
    "Selected signal policy: daily top 5%"
)
print("Minimum signals per date: 1")
print(
    "Total selected OOF signals:",
    int(policy_summary["signals"]),
)
print(
    "True positives:",
    int(policy_summary["true_positive"]),
)
print(
    "False positives:",
    int(policy_summary["false_positive"]),
)
print(
    "Precision:",
    float(policy_summary["precision"]),
)
print("Policy reselected in Stage 08: False")
print("Fixed probability threshold selected: False")
print("Hard started filter applied: False")
print("Hard breadth filter applied: False")
print("Economic returns used for selection: False")
print("Portfolio metrics used for selection: False")
print("Unseen-test loaded: False")
print("Unseen-test labels used: False")
print(
    "Next stage: apply this exact frozen policy once "
    "to the untouched unseen-test candidate period."
)


Notebook 08 integrity checks: PASSED
Selected model: xgboost
Selected feature set: I_full_40
Selected trial: 10
Raw feature count: 40
Score policy: raw identity ranking
Probability calibrator fitted: False
Literal probability interpretation allowed: False
Selected signal policy: daily top 5%
Minimum signals per date: 1
Total selected OOF signals: 3016
True positives: 1987
False positives: 1029
Precision: 0.6588196286472149
Policy reselected in Stage 08: False
Fixed probability threshold selected: False
Hard started filter applied: False
Hard breadth filter applied: False
Economic returns used for selection: False
Portfolio metrics used for selection: False
Unseen-test loaded: False
Unseen-test labels used: False
Next stage: apply this exact frozen policy once to the untouched unseen-test candidate period.


## Review before freezing Stage 08

Inspect:

- `results/audits/08_oof_input_audit.csv`
- `results/audits/08_signal_policy_summary.csv`
- `results/audits/08_signal_policy_fold_metrics.csv`
- `results/audits/08_signal_policy_date_audit.csv`
- `results/manifests/08_probability_signal_policy.json`
- `results/manifests/08_probability_signal_policy_manifest.json`
- local detailed output:
  `results/predictions/08_oof_signal_policy_predictions.csv`

Do not proceed to unseen-test inference until these policy artifacts are
audited and frozen.
